In [1]:
import os
from pathlib import Path

import numpy as np
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2 as T
from torchvision.models import ResNet18_Weights, resnet18

In [2]:
# for seed reproduction
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

batch_size = 16

id2label = {
	0: "glioma",
	1: "meningioma",
	2: "notumor",
	3: "pituitary",
}

root_path = Path.cwd().parent / "data"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
device

'cuda'

In [3]:
# !ls {root_path}

# 1. Data processing

In [4]:
class BrainTumorDataset(Dataset):
    def __init__(self, is_train: bool):
        super().__init__()

        self.is_train = is_train
        self.ds_path = root_path / f"{'train' if is_train else 'test'}"

        self.paths = os.listdir(self.ds_path)

        self.transforms = T.Compose([
            T.Resize((224, 224)),
            T.PILToTensor(),
            T.ToDtype(torch.float32, scale=True),
        ])

    def __getitem__(self, idx):
        # format: ./dataset/test/0040.jpg
        img_path = self.ds_path / f"{self.paths[idx]}"
        img = Image.open(img_path).convert("RGB")

        img = self.transforms(img)
        return img

    def __len__(self):
        return len(self.paths)

In [5]:
test_ds = BrainTumorDataset(is_train=False)
train_ds = BrainTumorDataset(is_train=True)

test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
# train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
csv = pd.read_csv(root_path / "train.csv")

train_labeled_ds = []
for idx, i in tqdm(enumerate(csv.values[:,0])):
    train_labeled_ds.append((train_ds[i], list(id2label.values()).index(csv.values[idx,1])))

train_labeled_ds;
train_loader = DataLoader(train_labeled_ds, batch_size=batch_size, shuffle=True)

48it [00:00, 385.57it/s]


In [6]:
# sanity check
x = next(iter(test_loader))
x.shape

torch.Size([16, 3, 224, 224])

# 2. Model

In [7]:
# you are not allowed to use any other pretrained model

# you need to have the internet turned on for the notebook, to download the weights (45MB)
resnet = resnet18(weights=ResNet18_Weights.DEFAULT).to(device)
resnet.fc = nn.Linear(512,4)
resnet.to(device);
for p in resnet.parameters():
    p.requires_grad_(True)

In [8]:
# sanity check
x = x.to(device)
y = resnet(x)
y.shape

torch.Size([16, 4])

In [9]:
optimizer = torch.optim.AdamW(resnet.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
epochs = 10

for epoch in range(epochs):
    resnet.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        # print(batch.shape)
        y_pred = resnet(batch_x)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # total_loss /= len(train_loader)
    # accuracy = 0
    # correct = 0
    # count = 0
    # resnet.eval()
    # with torch.no_grad():
    #     for batch_x, batch_y in test_loader:
    #         count += 1
    #         y_pred = resnet(batch).argmax()
    #         mask = batch_y == y_pred
    #         correct += mask.sum()
    #     accuracy = correct / count
    #     print(f"epoch: {epoch+1}\tloss: {total_loss}\taccuracy: {accuracy}")
    print(f"epoch: {epoch+1}\tloss: {total_loss}")

epoch: 1	loss: 4.351004123687744
epoch: 2	loss: 1.5382869839668274
epoch: 3	loss: 0.5308817327022552
epoch: 4	loss: 0.2529430389404297
epoch: 5	loss: 0.18600308150053024
epoch: 6	loss: 0.08435657992959023
epoch: 7	loss: 0.09086466021835804
epoch: 8	loss: 0.05345002934336662
epoch: 9	loss: 0.04245204012840986
epoch: 10	loss: 0.031990895979106426


# 3. Classification

Baseline approach: cluster images based on their resnet embeddings.

In [10]:
test_images = []

with torch.no_grad():
    for img in tqdm(test_loader):
        img = img.to(device)
        embeddings = resnet(img)
        test_images.append(embeddings.argmax(axis=1).detach().cpu())

test_images = torch.cat(test_images, dim=0)
test_images

100%|██████████| 82/82 [00:03<00:00, 22.83it/s]


tensor([2, 3, 1,  ..., 3, 0, 0])

In [11]:
# pca = PCA(n_components=2, random_state=seed)

# y_2d = pca.fit_transform(test_images.detach().cpu())

In [12]:
# km = KMeans(n_clusters=4, random_state=seed)

# clusters = km.fit_predict(y_2d)

In [13]:
# sns.scatterplot(
#     x=y_2d[:, 0], y=y_2d[:, 1],
#     hue=clusters,
#     palette="tab10",
#     legend="full"
# )
# plt.title("PCA of ResNet18 embeddings, colored by cluster ID")
# plt.show()

# 4. Submission

In [14]:
# # 1. find images closest to cluster centroids
# closest_idx = []
# for c in km.cluster_centers_:
#     dists = np.linalg.norm(y_2d - c, axis=1)
#     closest_idx.append(np.argmin(dists))

# # 2. plot them
# fig, axes = plt.subplots(2, 2, figsize=(6, 6))
# axes = axes.ravel()

# image_id2cluster = {i: int(c) for i, c in enumerate(clusters)}

# for ax, idx in zip(axes, closest_idx):
#     img_path = Path(test_ds.ds_path) / test_ds.paths[idx]
#     img = Image.open(img_path).convert("RGB")
#     ax.imshow(img)
#     ax.set_title(f"Cluster {image_id2cluster[idx]}")
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [15]:
"""
1. Is the scan empty?  
   - If the brain tissue looks normal and there is no extra mass, label it "notumor".

2. Is the mass inside the brain tissue itself?  
   - If yes, and it looks like a diffuse, ill-defined grey blob, label it "glioma".

3. Is the mass pushing on the brain from the outside edge (like a cap)?  
   - If yes, and it is round, well-defined, and often attached to the dura, label it "meningioma".

4. Is the mass sitting in the center of the head, between the two halves of the brain?  
   - If yes, and it is round and sitting in or above the pituitary area, label it "pituitary".
"""

id2label = {
	0: "glioma",
	1: "meningioma",
	2: "notumor",
	3: "pituitary",
}

# tip: you can also used the labeled samples, to avoid guessing

In [16]:
preds = [id2label[c.item()] for c in test_images]	
ids = [p[:len(".jpg")] for p in test_ds.paths]

In [17]:
sub = pd.DataFrame({"ID": ids, "prediction": preds})
sub.head()

,ID,prediction
0,0000,notumor
1,0001,pituitary
2,0002,meningioma
3,0003,glioma
4,0004,glioma


In [18]:
sub.to_csv(Path.cwd().parent / "submission.csv", index=False)